# Smart Product Pricing — Multimodal Fusion Notebook

Kaggle-ready end-to-end pipeline (preprocessing + training + inference) using TF-IDF, MiniLM, EfficientNet-B0 image embeddings, LightGBM, and Fusion MLP.

In [1]:
!pip install --quiet sentence-transformers timm lightgbm scikit-learn joblib

In [ ]:
import os, gc, json, random, re, hashlib, math, time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from PIL import Image
import timm

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
# DATA_DIR = Path("/kaggle/input/amazonml") if IN_KAGGLE else Path("68e8d1d70b66d_student_resource/student_resource/dataset")
# OUT_DIR = Path("//kaggle/working/") if IN_KAGGLE else Path("outputs")
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# (OUT_DIR / "features").mkdir(parents=True, exist_ok=True)
# (OUT_DIR / "cache").mkdir(parents=True, exist_ok=True)
# (OUT_DIR / "image_cache").mkdir(parents=True, exist_ok=True)
# (OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
# (OUT_DIR / "eval").mkdir(parents=True, exist_ok=True)

IN_KAGGLE = False

CONFIG = {
    "mini_eval_images": 400 if IN_KAGGLE else 0,
    "tfidf_max_features": 100_000,
    "brand_top_k": 1000,
    "text_batch": 256,
    "img_batch": 64,
    "img_workers": 2 if IN_KAGGLE else 0,
    "image_pca_dim": 256,
    "lgb_trials": 6 if IN_KAGGLE else 20,
    "lgb_num_boost_round": 2500 if IN_KAGGLE else 6000,
    "lgb_early_stopping": 150 if IN_KAGGLE else 200,
    "mlp_epochs": 14 if IN_KAGGLE else 40,
    "mlp_batch": 384 if IN_KAGGLE else 512,
    "mlp_patience": 4,
    "mlp_lr": 3e-3
}

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

train_path = "/Users/gabi/Desktop/DL stuff/AMAZONML/student_resource/dataset/train.csv"
test_path = "/Users/gabi/Desktop/DL stuff/AMAZONML/student_resource/dataset/test.csv"
if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError("Train/test CSVs missing. Attach dataset to notebook.")
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
print(f"Train shape: {train.shape} Test shape: {test.shape}")
print(f"Columns: {train.columns.tolist()}")
if "price" not in train:
    raise KeyError("price column missing in train.csv")
print("=== End Diagnostics ===")

=== Dataset Diagnostics ===
Kaggle mode: True
DATA_DIR: /kaggle/input/amazonml
Train shape: (75000, 4) Test shape: (75000, 3)
Columns: ['sample_id', 'catalog_content', 'image_link', 'price']
=== End Diagnostics ===


In [6]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("'", " ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

for col in ("catalog_content", "image_link"):
    if col in train:
        train[col] = train[col].fillna("")
    if col in test:
        test[col] = test[col].fillna("")
train["clean_text"] = train["catalog_content"].apply(clean_text)
test["clean_text"] = test["catalog_content"].apply(clean_text)

PACK_PATTERN = re.compile(r"\b(pack|pcs?|set|of|count|units?|ct|x)\b")
BRAND_MARKER_PATTERN = re.compile(r"\b(by|brand|make)\b")

def extract_first_float(text: str) -> float:
    if not isinstance(text, str):
        return np.nan
    match = re.search(r"\d+(?:\.\d+)?", text)
    return float(match.group()) if match else np.nan

train["ipq"] = train["clean_text"].apply(extract_first_float)
test["ipq"] = test["clean_text"].apply(extract_first_float)

for frame in (train, test):
    frame["text_len"] = frame["clean_text"].str.len()
    frame["num_count"] = frame["clean_text"].str.count(r"\d+")
    frame["digit_count"] = frame["clean_text"].str.count(r"\d")
    frame["has_digit"] = frame["digit_count"].gt(0).astype(np.float32)
    frame["has_pack_keywords"] = frame["clean_text"].apply(lambda s: 1.0 if PACK_PATTERN.search(s) else 0.0)
    frame["has_brand_marker"] = frame["clean_text"].apply(lambda s: 1.0 if BRAND_MARKER_PATTERN.search(s) else 0.0)
    frame["log_text_len"] = np.log1p(frame["text_len"].astype(float))
    frame["log_ipq"] = np.log1p(frame["ipq"].fillna(0.0).astype(float))

numeric_cols = [
    "ipq", "log_ipq", "text_len", "log_text_len", "num_count",
    "digit_count", "has_digit", "has_pack_keywords", "has_brand_marker"
]
num_train = train[numeric_cols].replace([np.inf, -np.inf], np.nan)
num_test = test[numeric_cols].replace([np.inf, -np.inf], np.nan)
medians = num_train.median().fillna(0.0)
num_train = num_train.fillna(medians)
num_test = num_test.fillna(medians)
scaler = StandardScaler()
num_train_scaled = scaler.fit_transform(num_train).astype(np.float32)
num_test_scaled = scaler.transform(num_test).astype(np.float32)

print(f"Numeric dim: {num_train_scaled.shape[1]}")

Numeric dim: 9


In [7]:
tfidf_vectorizer_path = OUT_DIR / "features" / "tfidf_vectorizer.joblib"
tfidf_train_path = OUT_DIR / "features" / "tfidf_train.npz"
tfidf_test_path = OUT_DIR / "features" / "tfidf_test.npz"

if tfidf_train_path.exists() and tfidf_test_path.exists() and tfidf_vectorizer_path.exists():
    tfidf_train = sp.load_npz(tfidf_train_path)
    tfidf_test = sp.load_npz(tfidf_test_path)
    tfidf_vectorizer = joblib.load(tfidf_vectorizer_path)
    print("Loaded cached TF-IDF")
else:
    tfidf_vectorizer = TfidfVectorizer(
        ngram_range=(1, 3),
        min_df=3,
        max_features=CONFIG["tfidf_max_features"],
        sublinear_tf=True,
        strip_accents="unicode"
    )
    tfidf_train = tfidf_vectorizer.fit_transform(train["clean_text"].tolist())
    tfidf_test = tfidf_vectorizer.transform(test["clean_text"].tolist())
    sp.save_npz(tfidf_train_path, tfidf_train)
    sp.save_npz(tfidf_test_path, tfidf_test)
    joblib.dump(tfidf_vectorizer, tfidf_vectorizer_path)
print(f"TF-IDF dim: {tfidf_train.shape[1]}")

BRAND_PATTERN = re.compile(r"^(?:by\s+|brand\s+)?([a-z0-9]{2,})")

def extract_brand(row: pd.Series) -> str:
    text = row.get("clean_text", "")
    match = BRAND_PATTERN.search(text)
    if match:
        return match.group(1)
    name = row.get("product_title") or row.get("product_name") or text
    if isinstance(name, str):
        name = re.sub(r"[^a-z0-9 ]", " ", name.lower())
        name = re.sub(r"\s+", " ", name).strip()
        if name:
            return name.split(" ")[0]
    return "unknown"

train["brand"] = train.apply(extract_brand, axis=1)
test["brand"] = test.apply(extract_brand, axis=1)
brand_counts = train["brand"].value_counts()
brand_vocab = brand_counts.head(CONFIG["brand_top_k"]).index.tolist()
brand_to_idx = {brand: idx for idx, brand in enumerate(brand_vocab)}

def build_brand_matrix(brands: pd.Series) -> sp.csr_matrix:
    rows, cols, data = [], [], []
    for i, brand in enumerate(brands):
        j = brand_to_idx.get(brand)
        if j is not None:
            rows.append(i)
            cols.append(j)
            data.append(1.0)
    return sp.csr_matrix((data, (rows, cols)), shape=(len(brands), len(brand_vocab)), dtype=np.float32)

brand_train = build_brand_matrix(train["brand"])
brand_test = build_brand_matrix(test["brand"])
sp.save_npz(OUT_DIR / "features" / "brand_train.npz", brand_train)
sp.save_npz(OUT_DIR / "features" / "brand_test.npz", brand_test)
joblib.dump({"vocab": brand_vocab}, OUT_DIR / "features" / "brand_encoder.joblib")
print(f"Brand dim: {brand_train.shape[1]}")

TF-IDF dim: 100000
Brand dim: 1


In [8]:
text_emb_train_path = OUT_DIR / "cache" / "text_emb_train.npy"
text_emb_test_path = OUT_DIR / "cache" / "text_emb_test.npy"

if text_emb_train_path.exists() and text_emb_test_path.exists():
    text_emb_train = np.load(text_emb_train_path)
    text_emb_test = np.load(text_emb_test_path)
    print("Loaded cached MiniLM embeddings")
else:
    text_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
    text_emb_train = text_model.encode(train["clean_text"].tolist(), batch_size=CONFIG["text_batch"], show_progress_bar=True, convert_to_numpy=True)
    text_emb_test = text_model.encode(test["clean_text"].tolist(), batch_size=CONFIG["text_batch"], show_progress_bar=True, convert_to_numpy=True)
    np.save(text_emb_train_path, text_emb_train.astype(np.float32))
    np.save(text_emb_test_path, text_emb_test.astype(np.float32))

text_emb_train = normalize(text_emb_train.astype(np.float32), norm="l2")
text_emb_test = normalize(text_emb_test.astype(np.float32), norm="l2")
np.save(text_emb_train_path, text_emb_train)
np.save(text_emb_test_path, text_emb_test)
print(f"MiniLM dim: {text_emb_train.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/293 [00:00<?, ?it/s]

Batches:   0%|          | 0/293 [00:00<?, ?it/s]

MiniLM dim: 384


In [9]:
from urllib.parse import urlparse

IMAGE_CACHE = OUT_DIR / "image_cache"

def url_to_filename(url: str) -> str:
    path = urlparse(url).path
    name = Path(path).name
    if name:
        return name
    return hashlib.md5(url.encode("utf-8")).hexdigest() + ".jpg"

def download_serial(url: str, dest: Path):
    import urllib.request
    try:
        urllib.request.urlretrieve(url, dest)
        return True
    except Exception:
        return False

def ensure_images(urls: List[str], limit: int = 0):
    try:
        from src.utils import download_images as bulk_download
    except Exception:
        bulk_download = None
    to_fetch = urls if limit == 0 else urls[:limit]
    missing = []
    for url in to_fetch:
        dest = IMAGE_CACHE / url_to_filename(url)
        if not dest.exists():
            missing.append((url, dest))
    if not missing:
        return
    if bulk_download is not None:
        try:
            bulk_download([u for u, _ in missing], str(IMAGE_CACHE))
            return
        except Exception:
            pass
    for url, dest in missing:
        download_serial(url, dest)

train_urls = train.get("image_link", []).astype(str).tolist()
test_urls = test.get("image_link", []).astype(str).tolist()
ensure_images(train_urls, CONFIG["mini_eval_images"])
ensure_images(test_urls, CONFIG["mini_eval_images"])

class ImageDataset(Dataset):
    def __init__(self, urls: List[str], limit: int = 0):
        self.urls = urls if limit == 0 else urls[:limit]
        self.paths = [IMAGE_CACHE / url_to_filename(url) for url in self.urls]

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        if not path.exists():
            return torch.zeros(3, 224, 224)
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            return torch.zeros(3, 224, 224)
        return transform(img)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
img_model.eval().to(device)
transform = timm.data.transforms_factory.create_transform(input_size=224, is_training=False)

def embed_images(urls: List[str], cache_path: Path) -> np.ndarray:
    if cache_path.exists():
        return np.load(cache_path)
    dataset = ImageDataset(urls, limit=CONFIG["mini_eval_images"])
    if len(dataset) == 0:
        emb = np.zeros((len(urls), img_model.num_features), dtype=np.float32)
        np.save(cache_path, emb)
        return emb
    loader = DataLoader(
        dataset,
        batch_size=CONFIG["img_batch"],
        shuffle=False,
        num_workers=CONFIG["img_workers"],
        pin_memory=device.type == "cuda"
    )
    outputs: List[np.ndarray] = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            feats = img_model(batch).cpu().numpy().astype(np.float32)
            outputs.append(feats)
    emb = np.concatenate(outputs, axis=0)
    if emb.shape[0] < len(urls):
        pad = np.zeros((len(urls) - emb.shape[0], emb.shape[1]), dtype=np.float32)
        emb = np.vstack([emb, pad])
    np.save(cache_path, emb)
    return emb

image_emb_train = embed_images(train_urls, OUT_DIR / "cache" / "image_emb_train.npy")
image_emb_test = embed_images(test_urls, OUT_DIR / "cache" / "image_emb_test.npy")
print(f"Image emb dim: {image_emb_train.shape[1] if image_emb_train.ndim == 2 else 0}")

pca_path = OUT_DIR / "features" / "image_pca.joblib"
if pca_path.exists():
    image_pca_model = joblib.load(pca_path)
else:
    image_pca_model = PCA(n_components=CONFIG["image_pca_dim"], random_state=SEED)
    image_pca_model.fit(image_emb_train)
    joblib.dump(image_pca_model, pca_path)
image_pca_train = image_pca_model.transform(image_emb_train).astype(np.float32)
image_pca_test = image_pca_model.transform(image_emb_test).astype(np.float32)
print(f"Image PCA dim: {image_pca_train.shape[1]}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Image emb dim: 1280
Image PCA dim: 256


In [ ]:
dense_train = np.hstack([text_emb_train, image_pca_train, num_train_scaled]).astype(np.float32)
dense_test = np.hstack([text_emb_test, image_pca_test, num_test_scaled]).astype(np.float32)
sparse_train = sp.hstack([tfidf_train.astype(np.float32), brand_train], format="csr")
sparse_test = sp.hstack([tfidf_test.astype(np.float32), brand_test], format="csr")

np.save(OUT_DIR / "features" / "dense_train.npy", dense_train)
np.save(OUT_DIR / "features" / "dense_test.npy", dense_test)
sp.save_npz(OUT_DIR / "features" / "sparse_train.npz", sparse_train)
sp.save_npz(OUT_DIR / "features" / "sparse_test.npz", sparse_test)

manifest = {
    "paths": {
        "dense_train": str(OUT_DIR / "features" / "dense_train.npy"),
        "dense_test": str(OUT_DIR / "features" / "dense_test.npy"),
        "sparse_train": str(OUT_DIR / "features" / "sparse_train.npz"),
        "sparse_test": str(OUT_DIR / "features" / "sparse_test.npz"),
        "text_emb_train": str(OUT_DIR / "cache" / "text_emb_train.npy"),
        "text_emb_test": str(OUT_DIR / "cache" / "text_emb_test.npy"),
        "image_emb_train": str(OUT_DIR / "cache" / "image_emb_train.npy"),
        "image_emb_test": str(OUT_DIR / "cache" / "image_emb_test.npy"),
        "image_pca_train": str(OUT_DIR / "cache" / "image_pca_train.npy"),
        "image_pca_test": str(OUT_DIR / "cache" / "image_pca_test.npy"),
        "numeric_train": str(OUT_DIR / "cache" / "numeric_train.npy"),
        "numeric_test": str(OUT_DIR / "cache" / "numeric_test.npy"),
        "numeric_scaler": str(OUT_DIR / "features" / "numeric_scaler.joblib"),
        "tfidf_vectorizer": str(tfidf_vectorizer_path),
        "brand_encoder": str(OUT_DIR / "features" / "brand_encoder.joblib"),
        "train_csv": str(train_path),
        "test_csv": str(test_path)
    },
    "shapes": {
        "dense": dense_train.shape,
        "sparse": sparse_train.shape,
        "text_emb": text_emb_train.shape,
        "image_pca": image_pca_train.shape,
        "numeric": num_train_scaled.shape
    }
}
joblib.dump({"scaler": scaler, "columns": numeric_cols}, OUT_DIR / "features" / "numeric_scaler.joblib")
(OUT_DIR / "features" / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Dense dim:", dense_train.shape[1])
print("Sparse dim:", sparse_train.shape[1])

In [ ]:
y_price = train["price"].astype(float).to_numpy()
y_log = np.log1p(y_price)
sample_weights = np.clip(1.0 / np.power(y_price, 0.5), 0.5, 5.0).astype(np.float32)
num_bins = min(20, max(5, len(y_log) // 500))
bins = pd.qcut(y_log, q=num_bins, labels=False, duplicates="drop")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(len(y_log)), bins))
print("Folds ready ->", len(folds))

train_matrix = sp.hstack([sparse_train, sp.csr_matrix(dense_train)], format="csr", dtype=np.float32)
test_matrix = sp.hstack([sparse_test, sp.csr_matrix(dense_test)], format="csr", dtype=np.float32)
print(f"Combined feature dim: {train_matrix.shape[1]}")

def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denom = np.where(denom == 0.0, 1.0, denom)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)

def smape_eval(preds: np.ndarray, dataset: lgb.Dataset):
    labels = dataset.get_label()
    score = smape(np.expm1(labels), np.expm1(preds))
    return "smape", score, False

print("=== LightGBM Hyperparameter Search ===")
base_params = {
    "objective": "regression",
    "metric": "None",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.65,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 2.0,
    "max_bin": 255,
    "seed": SEED
}

def sample_params(rng: np.random.Generator) -> Dict[str, float]:
    params = base_params.copy()
    params["num_leaves"] = int(rng.integers(31, 128))
    params["min_data_in_leaf"] = int(rng.integers(20, 201))
    params["feature_fraction"] = float(rng.uniform(0.5, 0.9))
    params["bagging_fraction"] = float(rng.uniform(0.6, 0.9))
    params["lambda_l2"] = float(rng.uniform(0.0, 10.0))
    params["learning_rate"] = float(rng.uniform(0.01, 0.07))
    return params

rng = np.random.default_rng(SEED)
search_folds = folds[:min(3, len(folds))]
best_params = base_params.copy()
best_score = float("inf")

for trial in range(CONFIG["lgb_trials"]):
    params = sample_params(rng)
    trial_scores = []
    for tr_idx, val_idx in search_folds:
        lgb_train = lgb.Dataset(train_matrix[tr_idx], label=y_log[tr_idx], weight=sample_weights[tr_idx], free_raw_data=False)
        lgb_valid = lgb.Dataset(train_matrix[val_idx], label=y_log[val_idx], weight=sample_weights[val_idx], free_raw_data=False)
        booster = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_valid],
            feval=smape_eval,
            num_boost_round=CONFIG["lgb_num_boost_round"],
            callbacks=[lgb.early_stopping(CONFIG["lgb_early_stopping"], verbose=False), lgb.log_evaluation(0)]
        )
        preds_val = booster.predict(train_matrix[val_idx], num_iteration=booster.best_iteration or booster.current_iteration())
        trial_scores.append(smape(np.expm1(y_log[val_idx]), np.expm1(preds_val)))
    mean_score = float(np.mean(trial_scores))
    print(f"Trial {trial+1:02d} SMAPE {mean_score:.4f}")
    if mean_score < best_score:
        best_score = mean_score
        best_params = params

print("Best params:", best_params, "score=", best_score)

Folds ready -> 5
Combined feature dim: 100650
=== LightGBM Hyperparameter Search ===


In [ ]:
lgb_dir = OUT_DIR / "models" / "lightgbm"
lgb_dir.mkdir(parents=True, exist_ok=True)
lgb_oof = np.zeros_like(y_price, dtype=np.float32)
lgb_test_preds: List[np.ndarray] = []
lgb_fold_scores: List[float] = []

for fold_id, (tr_idx, val_idx) in enumerate(folds):
    lgb_train = lgb.Dataset(train_matrix[tr_idx], label=y_log[tr_idx], weight=sample_weights[tr_idx], free_raw_data=False)
    lgb_valid = lgb.Dataset(train_matrix[val_idx], label=y_log[val_idx], weight=sample_weights[val_idx], free_raw_data=False)
    booster = lgb.train(
        best_params,
        lgb_train,
        valid_sets=[lgb_valid],
        feval=smape_eval,
        num_boost_round=CONFIG["lgb_num_boost_round"],
        callbacks=[lgb.early_stopping(CONFIG["lgb_early_stopping"], verbose=False), lgb.log_evaluation(100)]
    )
    best_iter = booster.best_iteration or booster.current_iteration()
    val_preds_log = booster.predict(train_matrix[val_idx], num_iteration=best_iter)
    val_preds = np.expm1(val_preds_log).clip(0.01)
    fold_score = smape(y_price[val_idx], val_preds)
    lgb_fold_scores.append(float(fold_score))
    lgb_oof[val_idx] = val_preds.astype(np.float32)
    booster.save_model(str(lgb_dir / f"fold_{fold_id}.txt"), num_iteration=best_iter)
    test_preds_log = booster.predict(test_matrix, num_iteration=best_iter)
    lgb_test_preds.append(np.expm1(test_preds_log).clip(0.01).astype(np.float32))
    print(f"Fold {fold_id} SMAPE {fold_score:.4f} iter {best_iter}")

print("LightGBM fold SMAPE:", lgb_fold_scores)

In [ ]:
# Early OOF save: write current LightGBM OOF and test preds so you can inspect progress without waiting for full run
import numpy as np
from pathlib import Path
(oof_parent := (OUT_DIR / 'eval')).mkdir(parents=True, exist_ok=True)
oof_path = OUT_DIR / 'eval' / 'lgb_oof_partial.npy'
test_preds_path = OUT_DIR / 'eval' / 'lgb_test_partial.npy'
# lgb_oof and lgb_test_preds are populated during folds; save safely if available
try:
    if 'lgb_oof' in globals():
        np.save(oof_path, lgb_oof)
    if 'lgb_test_preds' in globals() and len(lgb_test_preds):
        np.save(test_preds_path, np.vstack(lgb_test_preds))
    else:
        np.save(test_preds_path, np.zeros((0,)))
    print('Saved partial OOF ->', oof_path)
    print('Saved partial test preds ->', test_preds_path)
except Exception as _e:
    print('Could not save partial preds yet:', _e)
# Also write a tiny CV JSON so you can see fold scores so far
cv_json = OUT_DIR / 'eval' / 'cv_partial.json'
import json
json.dump({'lgb_fold_scores': globals().get('lgb_fold_scores', []), 'mean_smape': float(np.mean(globals().get('lgb_fold_scores', []))) if len(globals().get('lgb_fold_scores', []))>0 else None}, open(cv_json, 'w'))
print('Wrote partial CV metrics ->', cv_json)


In [ ]:
class DenseDataset(Dataset):
    def __init__(self, feats: np.ndarray, targets: np.ndarray, weights: np.ndarray):
        self.feats = torch.from_numpy(feats.astype(np.float32))
        self.targets = torch.from_numpy(targets.astype(np.float32))
        self.weights = torch.from_numpy(weights.astype(np.float32))

    def __len__(self) -> int:
        return self.feats.shape[0]

    def __getitem__(self, idx: int):
        return self.feats[idx], self.targets[idx], self.weights[idx]

class FusionRegressor(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
mlp_dir = OUT_DIR / "models" / "fusion_mlp"
mlp_dir.mkdir(parents=True, exist_ok=True)
mlp_oof = np.zeros_like(y_price, dtype=np.float32)
mlp_test_preds: List[np.ndarray] = []
mlp_fold_scores: List[float] = []
criterion = nn.SmoothL1Loss(reduction="none")
test_dataset = DenseDataset(dense_test, np.zeros(dense_test.shape[0], dtype=np.float32), np.ones(dense_test.shape[0], dtype=np.float32))
test_loader = DataLoader(test_dataset, batch_size=CONFIG["mlp_batch"], shuffle=False, num_workers=0, pin_memory=use_amp)

for fold_id, (tr_idx, val_idx) in enumerate(folds):
    model = FusionRegressor(dense_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["mlp_lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["mlp_epochs"], eta_min=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    train_dataset = DenseDataset(dense_train[tr_idx], y_log[tr_idx], sample_weights[tr_idx])
    val_dataset = DenseDataset(dense_train[val_idx], y_log[val_idx], sample_weights[val_idx])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["mlp_batch"], shuffle=True, num_workers=0, pin_memory=use_amp)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["mlp_batch"], shuffle=False, num_workers=0, pin_memory=use_amp)

    best_state = None
    best_score = float("inf")
    patience = 0

    for epoch in range(CONFIG["mlp_epochs"]):
        model.train()
        for xb, yb, wb in train_loader:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                preds = model(xb)
                loss = (criterion(preds, yb) * wb).mean()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        model.eval()
        val_preds = []
        with torch.no_grad():
            for xb, _, _ in val_loader:
                xb = xb.to(device)
                preds = model(xb)
                val_preds.append(preds.cpu().numpy())
        val_pred_log = np.concatenate(val_preds).reshape(-1)
        val_pred_price = np.expm1(val_pred_log).clip(0.01)
        fold_score = smape(y_price[val_idx], val_pred_price)
        print(f"[Fusion] Fold {fold_id} epoch {epoch+1:02d} SMAPE={fold_score:.4f}")
        if fold_score < best_score - 1e-6:
            best_score = fold_score
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
        if patience >= CONFIG["mlp_patience"]:
            break

    if best_state is None:
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state, strict=True)
    torch.save(best_state, mlp_dir / f"fold_{fold_id}.pt")
    model.to(device)

    model.eval()
    val_preds = []
    with torch.no_grad():
        for xb, _, _ in val_loader:
            xb = xb.to(device)
            preds = model(xb)
            val_preds.append(preds.cpu().numpy())
    val_pred_log = np.concatenate(val_preds).reshape(-1)
    val_pred_price = np.expm1(val_pred_log).clip(0.01)
    mlp_oof[val_idx] = val_pred_price.astype(np.float32)
    final_score = smape(y_price[val_idx], val_pred_price)
    mlp_fold_scores.append(float(final_score))
    print(f"[Fusion] Fold {fold_id} final SMAPE={final_score:.4f}")

    test_preds = []
    with torch.no_grad():
        for xb, _, _ in test_loader:
            xb = xb.to(device)
            preds = model(xb)
            test_preds.append(preds.cpu().numpy())
    test_pred_log = np.concatenate(test_preds).reshape(-1)
    mlp_test_preds.append(np.expm1(test_pred_log).clip(0.01).astype(np.float32))
    gc.collect()

print("Fusion fold SMAPE:", mlp_fold_scores)

In [ ]:
p1, p995 = np.percentile(y_price, [1, 99.5])
lgb_oof_clipped = np.clip(lgb_oof, p1, p995)
mlp_oof_clipped = np.clip(mlp_oof, p1, p995)
lgb_test_mean = np.clip(np.mean(np.stack(lgb_test_preds), axis=0), p1, p995)
mlp_test_mean = np.clip(np.mean(np.stack(mlp_test_preds), axis=0), p1, p995)

weights_grid = [0.3, 0.4, 0.5, 0.6, 0.7]
ensemble_scores: Dict[float, float] = {}
for w in weights_grid:
    ens_pred = w * lgb_oof_clipped + (1.0 - w) * mlp_oof_clipped
    ensemble_scores[w] = smape(y_price, ens_pred)
best_weight = min(ensemble_scores, key=ensemble_scores.get)
ensemble_oof = np.clip(best_weight * lgb_oof_clipped + (1.0 - best_weight) * mlp_oof_clipped, p1, p995)
ensemble_test = np.clip(best_weight * lgb_test_mean + (1.0 - best_weight) * mlp_test_mean, p1, p995)
ensemble_test = np.clip(ensemble_test, 0.01, None)

ensemble_fold_scores = []
for _, val_idx in folds:
    fold_pred = best_weight * lgb_oof_clipped[val_idx] + (1.0 - best_weight) * mlp_oof_clipped[val_idx]
    ensemble_fold_scores.append(smape(y_price[val_idx], fold_pred))

metrics_payload = {
    "models": ["lightgbm", "fusion_mlp", "ensemble"],
    "fold_smape": {
        "lightgbm": list(map(float, lgb_fold_scores)),
        "fusion_mlp": list(map(float, mlp_fold_scores)),
        "ensemble": list(map(float, ensemble_fold_scores))
    },
    "oof_smape": {
        "lightgbm": float(smape(y_price, lgb_oof_clipped)),
        "fusion_mlp": float(smape(y_price, mlp_oof_clipped)),
        "ensemble": float(smape(y_price, ensemble_oof))
    },
    "ensemble_weight": float(best_weight),
    "search_scores": {str(k): float(v) for k, v in ensemble_scores.items()}
}
(OUT_DIR / "eval" / "cv_metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
(OUT_DIR / "ensemble.json").write_text(json.dumps({"weight": float(best_weight)}, indent=2), encoding="utf-8")

print(f"LightGBM OOF SMAPE: {metrics_payload['oof_smape']['lightgbm']:.4f}")
print(f"Fusion MLP OOF SMAPE: {metrics_payload['oof_smape']['fusion_mlp']:.4f}")
print(f"Ensemble OOF SMAPE: {metrics_payload['oof_smape']['ensemble']:.4f}")
print("Ensemble weight grid:", ensemble_scores)
print("Best weight:", best_weight)
print(f"Improvement vs baseline 63.5955: {63.5955 - metrics_payload['oof_smape']['ensemble']:.4f}")

In [ ]:
oof_df = pd.DataFrame({
    "sample_id": train["sample_id"],
    "price_true": y_price,
    "lgb_pred": lgb_oof,
    "mlp_pred": mlp_oof,
    "ensemble_pred": ensemble_oof
})
oof_df.to_csv(OUT_DIR / "eval" / "oof_preds.csv", index=False)

test_out = test[["sample_id"]].copy()
test_out["price"] = ensemble_test
assert len(test_out) == len(test)
assert (test_out["price"] > 0).all()
test_out.to_csv(OUT_DIR / "test_out.csv", index=False)

print("Saved:")
print("- outputs/test_out.csv")
print("- outputs/eval/cv_metrics.json")
print("- outputs/eval/oof_preds.csv")
print("License note: Predictions derived solely from provided data (Apache/MIT compliant models).")